In [1]:
import math
import cvxpy as cp
import numpy as np
from functions import load_scenarios_with_flexible
import argparse
import csv
import statistics
from typing import List, Dict, Any
import pickle
import os
import random

In [2]:
def open_results(trace="CAISO", 
                 T=48, 
                 gamma=10.0, 
                 delta=5.0, 
                 c_delivery=0.2, 
                 eps_delivery=0.05, 
                 proportion_base=0.5, 
                 scale_factor=40.0):
    
    filename = f'eval_results/{trace}_T{T}_gamma{gamma}_delta{delta}_c{c_delivery}_eps{eps_delivery}_prop{proportion_base}_scale{scale_factor}.pkl'
    with open(filename, 'rb') as f:
        results = pickle.load(f)
    return results

In [3]:
def summarize(values: List[float]) -> Dict[str, float]:
    if not values:
        return {}
    vs = sorted(values)
    def pct(p): return vs[int(p * (len(vs) - 1))]
    return {
        "mean": float(sum(vs) / len(vs)),
        "median": float(statistics.median(vs)),
        "p10": pct(0.10),
        "p25": pct(0.25),
        "p75": pct(0.75),
        "p95": pct(0.95),
        "min": vs[0],
        "max": vs[-1],
    }

def print_summary(label, vals):
    if not vals:
        print(f"{label}: (none)")
        return
    s = summarize(vals)
    print(f"{label}:    mean={s['mean']:.4f}    median={s['median']:.4f}    p95={s['p95']:.4f}    min={s['min']:.4f}")


In [4]:
rows = open_results()

ratios_pald = [r["pald_over_opt"] for r in rows if r.get("pald_over_opt")]
ratios_paad = [r["paad_over_opt"] for r in rows if r.get("paad_over_opt")]
# truncate ratios to 1.0
ratios_pald = [max(1.0, r) for r in ratios_pald]
ratios_paad = [max(1.0, r) for r in ratios_paad]
print_summary("PALD/OPT", ratios_pald)
print_summary("PAAD/OPT", ratios_paad)

PALD/OPT:    mean=1.4385    median=1.3452    p95=2.1022    min=1.0000
PAAD/OPT:    mean=1.6484    median=1.5559    p95=2.3026    min=1.1231


In [6]:
scale_factors = [40.0, 80.0, 160.0, 320.0, 480.0, 640.0]
for scale in scale_factors:
    rows = open_results(scale_factor=scale)
    ratios_pald = [r["pald_over_opt"] for r in rows if r.get("pald_over_opt")]
    ratios_paad = [r["paad_over_opt"] for r in rows if r.get("paad_over_opt")]
    # truncate ratios to 1.0
    ratios_pald = [max(1.0, r) for r in ratios_pald]
    ratios_paad = [max(1.0, r) for r in ratios_paad]
    print(f"Scale factor: {scale}")
    print_summary("  PALD/OPT", ratios_pald)
    print_summary("  PAAD/OPT", ratios_paad)
    print()

Scale factor: 40.0
  PALD/OPT:    mean=1.4385    median=1.3452    p95=2.1022    min=1.0000
  PAAD/OPT:    mean=1.6484    median=1.5559    p95=2.3026    min=1.1231

Scale factor: 80.0
  PALD/OPT:    mean=1.5994    median=1.4246    p95=2.6882    min=1.0222
  PAAD/OPT:    mean=1.7505    median=1.6158    p95=2.6048    min=1.1027

Scale factor: 160.0
  PALD/OPT:    mean=1.8569    median=1.5417    p95=3.6319    min=1.0686
  PAAD/OPT:    mean=1.8338    median=1.6601    p95=2.9975    min=1.0935

Scale factor: 320.0
  PALD/OPT:    mean=2.3594    median=1.8029    p95=5.4352    min=1.0833
  PAAD/OPT:    mean=1.9013    median=1.6661    p95=3.5317    min=1.0897

Scale factor: 480.0
  PALD/OPT:    mean=2.8463    median=2.0636    p95=6.9119    min=1.1068
  PAAD/OPT:    mean=1.9094    median=1.6428    p95=3.6786    min=1.0886

Scale factor: 640.0
  PALD/OPT:    mean=3.3399    median=2.3411    p95=8.3761    min=1.1025
  PAAD/OPT:    mean=1.9041    median=1.6261    p95=3.7949    min=1.0882

